# Cerrar sesión del MCP (liberar cupo de sesiones)

Este notebook tiene **un solo propósito**: cerrar la sesión que quedó abierta
en el MCP por culpa de nuestros propios scripts/notebooks (`mcp_client`
cachea el token en `.mcp_token_cache.json` y lo reutiliza en vez de
loguearse cada vez — eso es justo lo que te dejó sin cupo para entrar por la
GUI, porque tu cuenta tiene `concurrentSessionMax` limitado).

**Importante — qué SÍ y qué NO puede hacer este notebook:**

- ✅ Puede cerrar la sesión asociada al token cacheado localmente (la que
  crearon `main.py`/`ciena_mcp_tutorial.ipynb` en este equipo), usando ese
  mismo token — **sin necesidad de loguearse de nuevo**. Por eso funciona
  aunque ya estés en el límite de sesiones: no pide una sesión nueva, cierra
  una que ya existe.
- ❌ NO puede listar ni cerrar sesiones de otro origen (ej. una pestaña del
  navegador que quedó abierta) — el endpoint `/tron/api/v1/sessions` está
  restringido para tu rol (`403 RESTRICTED and DENIED`). Si el problema
  persiste después de correr este notebook, la otra sesión "atascada" es de
  la GUI y hay que cerrarla desde ahí, o pedirle a un administrador de
  Security/tron (rol con acceso a `/tron/api/v1/sessions`) que la libere.


In [ ]:
import json
from pathlib import Path

TOKEN_CACHE_PATH = Path(".mcp_token_cache.json")

if not TOKEN_CACHE_PATH.exists():
    print("No hay token cacheado localmente (.mcp_token_cache.json no existe).")
    print("Este notebook no tiene ninguna sesión propia que cerrar.")
    cached = None
else:
    cached = json.loads(TOKEN_CACHE_PATH.read_text())
    print("Token cacheado encontrado para usuario:", cached["username"])


## Confirmar qué sesión se va a cerrar

Antes de cerrar nada, consultamos `/tron/api/v1/current-user` con ese token
para ver el detalle de la sesión (IP, hora, tipo) y confirmar que es la que
queremos liberar.


In [ ]:
import os
import requests
import urllib3
from dotenv import load_dotenv

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
load_dotenv()

base_url = f"https://{os.environ['MCP_SERVER_IP']}"

if cached:
    headers = {"Authorization": f"Bearer {cached['token']}", "accept": "application/json"}
    r = requests.get(f"{base_url}/tron/api/v1/current-user", headers=headers, verify=False, timeout=15)
    if r.status_code == 200:
        detail = r.json()["lastLoginDetail"]
        print("Sesión activa a cerrar:")
        print(json.dumps(detail, indent=2, ensure_ascii=False))
    else:
        print(f"El token cacheado ya no es válido (status {r.status_code}) — probablemente ya expiró o se cerró.")
        print(r.text)
        cached = None


## Cerrar la sesión

`POST /tron/api/v1/logout` con el token cacheado. Esto libera el cupo de
sesión sin necesidad de un login nuevo.


In [ ]:
if cached:
    r = requests.post(
        f"{base_url}/tron/api/v1/logout",
        headers={"Authorization": f"Bearer {cached['token']}"},
        verify=False,
        timeout=15,
    )
    print("logout status:", r.status_code)

    if r.status_code in (200, 204):
        TOKEN_CACHE_PATH.unlink()
        print("Sesión cerrada y cache local borrado (.mcp_token_cache.json).")
        print("La próxima vez que corras main.py o el tutorial, mcp_client hará login de nuevo.")
    else:
        print("No se pudo cerrar la sesión:", r.text)
else:
    print("Nada que cerrar.")


## Si el problema persiste

Si después de correr este notebook sigues viendo
`"Maximum number of active sessions reached"` al entrar por la GUI, significa
que la otra sesión activa no es la de este notebook (por ejemplo, una
pestaña del navegador que quedó abierta en otro equipo). En ese caso:

1. Revisa si tienes otra sesión de navegador abierta en otro dispositivo y
   ciérrala desde ahí.
2. Si no la encuentras, pide a un administrador con acceso a
   `/tron/api/v1/sessions` (rol de Security/tron) que la termine manualmente.
